In [1]:
import numpy as np
import pandas as pd

In [2]:
data_ids = [361254, 361259, 361253, 361242]
n_ests = [50, 100, 500, 1000]
min_samples_leafs = [1, 5, 10]
max_features = [0.1, 0.33, "1.0"]

In [3]:
# for each data_id, load the result and save as a df
dfs = []
for data_id in data_ids:
    # get number of samples in the data_id by reading X csv
    X = np.loadtxt(f"data/{data_id}/X.csv", delimiter=",")
    n_samples = 2000
    if X.shape[0] < n_samples:
        n_samples = X.shape[0]
    n_features = X.shape[1]
    for n_est in n_ests:
        for min_samples_leaf in min_samples_leafs:
            for max_feature in max_features:
                # create the directory if it doesn't exist
                dir_path = f"results/rf/{data_id}/n_estimators_{n_est}/min_samples_leaf_{min_samples_leaf}/max_features_{max_feature}"
                results_path = f"{dir_path}/runtime_results.csv"
                results_df = pd.read_csv(results_path)
                # divide every col in df except 'data_id' by n_samples
                for col in results_df.columns:
                    if col != 'data_id':
                        results_df[col] = results_df[col] / n_samples
                # add columns for n_estimators, min_samples_leaf, max_features
                results_df['n_estimators'] = n_est
                results_df['min_samples_leaf'] = min_samples_leaf
                results_df['max_features'] = max_feature
                results_df['num_features'] = n_features
                dfs.append(results_df)
df = pd.concat(dfs, ignore_index=True)

In [4]:
df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])

,data_id,rf_fitting_time,rf_plus_fitting_time,shap_rf_explainer_time,shap_rf_values_time,lime_rf_time,local_mdi_time,lmdi_plus_rf_explainer_time,lmdi_plus_rf_values_time,n_estimators,min_samples_leaf,max_features,num_features
118,361242,0.001803,0.045238,0.000024,0.090662,0.256854,0.001446,0.000006,0.104910,100,1,0.33,81
82,361253,0.001368,0.040014,0.000012,0.094738,0.204851,0.001523,0.000007,0.063547,100,1,0.33,48
10,361254,0.000640,0.042388,0.000020,0.062672,0.132276,0.001638,0.000005,0.032119,100,1,0.33,21
46,361259,0.001074,0.081427,0.000026,0.085067,0.173325,0.001650,0.000005,0.042685,100,1,0.33,32
121,361242,0.001188,0.007658,0.000006,0.009323,0.235120,0.001091,0.000004,0.037112,100,5,0.33,81
85,361253,0.000982,0.005556,0.000006,0.009463,0.178868,0.001081,0.000004,0.017126,100,5,0.33,48
13,361254,0.000416,0.005744,0.000007,0.006780,0.105039,0.001075,0.000003,0.005571,100,5,0.33,21
49,361259,0.000786,0.007079,0.000007,0.008857,0.142327,0.001263,0.000004,0.010444,100,5,0.33,32
124,361242,0.001040,0.005316,0.000005,0.003255,0.227542,0.000885,0.000003,0.026444,100,10,0.33,81
88,361253,0.000898,0.004366,0.000005,0.003813,0.171565,0.000907,0.000003,0.012139,100,10,0.33,48


In [5]:
display_df = df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, rf_plus_fitting_time + lmdi_plus_values_time, lime_time, shap_values_time, local_mdi_time
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'rf_plus_fitting_time', 'lmdi_plus_rf_values_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_rf_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_rf_values_time'], inplace=True)
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'lmdi_plus_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lmdi_plus_time': 'LMDI+',
    'lime_rf_time': 'LIME',
    'shap_rf_values_time': 'TreeSHAP',
    'local_mdi_time': 'Local MDI'
})

# sort display_df by min samples leaf increasing and then number of features increasing
display_df = display_df.sort_values(by=['Min. Samples per Leaf', '# of Features'])

# round to fourth decimal place
display_df = display_df.round(4)

In [6]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   Min. Samples per Leaf |   LMDI+ |   LIME |   TreeSHAP |   Local MDI |
|-----------------:|----------------:|------------------------:|--------:|-------:|-----------:|------------:|
|           361254 |              21 |                       1 |  0.0745 | 0.1323 |     0.0627 |      0.0016 |
|           361259 |              32 |                       1 |  0.1241 | 0.1733 |     0.0851 |      0.0016 |
|           361253 |              48 |                       1 |  0.1036 | 0.2049 |     0.0947 |      0.0015 |
|           361242 |              81 |                       1 |  0.1501 | 0.2569 |     0.0907 |      0.0014 |
|           361254 |              21 |                       5 |  0.0113 | 0.105  |     0.0068 |      0.0011 |
|           361259 |              32 |                       5 |  0.0175 | 0.1423 |     0.0089 |      0.0013 |
|           361253 |              48 |                       5 |  0.0227 | 0.1789 |     0.0095 |      0.0011 |
|

In [7]:
# get display_df in latex format, still only use 4 decimal places
latex_df = display_df.to_latex(index=False, float_format="%.4f")
print(latex_df)

\begin{tabular}{rrrrrrr}
\toprule
OpenML Data ID & # of Features & Min. Samples per Leaf & LMDI+ & LIME & TreeSHAP & Local MDI \\
\midrule
361254 & 21 & 1 & 0.0745 & 0.1323 & 0.0627 & 0.0016 \\
361259 & 32 & 1 & 0.1241 & 0.1733 & 0.0851 & 0.0016 \\
361253 & 48 & 1 & 0.1036 & 0.2049 & 0.0947 & 0.0015 \\
361242 & 81 & 1 & 0.1501 & 0.2569 & 0.0907 & 0.0014 \\
361254 & 21 & 5 & 0.0113 & 0.1050 & 0.0068 & 0.0011 \\
361259 & 32 & 5 & 0.0175 & 0.1423 & 0.0089 & 0.0013 \\
361253 & 48 & 5 & 0.0227 & 0.1789 & 0.0095 & 0.0011 \\
361242 & 81 & 5 & 0.0448 & 0.2351 & 0.0093 & 0.0011 \\
361254 & 21 & 10 & 0.0081 & 0.0993 & 0.0028 & 0.0010 \\
361259 & 32 & 10 & 0.0128 & 0.1332 & 0.0031 & 0.0010 \\
361253 & 48 & 10 & 0.0165 & 0.1716 & 0.0038 & 0.0009 \\
361242 & 81 & 10 & 0.0318 & 0.2275 & 0.0033 & 0.0009 \\
\bottomrule
\end{tabular}



In [8]:
df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5)].sort_values(by=['n_estimators', 'data_id'])

,data_id,rf_fitting_time,rf_plus_fitting_time,shap_rf_explainer_time,shap_rf_values_time,lime_rf_time,local_mdi_time,lmdi_plus_rf_explainer_time,lmdi_plus_rf_values_time,n_estimators,min_samples_leaf,max_features,num_features
112,361242,0.000634,0.005010,0.000004,0.004434,0.213647,0.000544,0.000002,0.018382,50,5,0.33,81
76,361253,0.000472,0.003702,0.000004,0.004706,0.150618,0.000550,0.000002,0.008597,50,5,0.33,48
4,361254,0.000216,0.003724,0.000002,0.003272,0.072877,0.000552,0.000002,0.003634,50,5,0.33,21
40,361259,0.000369,0.004650,0.000005,0.004270,0.113425,0.000608,0.000002,0.005944,50,5,0.33,32
121,361242,0.001188,0.007658,0.000006,0.009323,0.235120,0.001091,0.000004,0.037112,100,5,0.33,81
85,361253,0.000982,0.005556,0.000006,0.009463,0.178868,0.001081,0.000004,0.017126,100,5,0.33,48
13,361254,0.000416,0.005744,0.000007,0.006780,0.105039,0.001075,0.000003,0.005571,100,5,0.33,21
49,361259,0.000786,0.007079,0.000007,0.008857,0.142327,0.001263,0.000004,0.010444,100,5,0.33,32
130,361242,0.006469,0.040497,0.000033,0.044747,0.380018,0.005128,0.000021,0.185142,500,5,0.33,81
94,361253,0.004757,0.032215,0.000032,0.048578,0.341644,0.005396,0.000020,0.084858,500,5,0.33,48


In [9]:
display_df

,OpenML Data ID,# of Features,Min. Samples per Leaf,LMDI+,LIME,TreeSHAP,Local MDI
10,361254,21,1,0.0745,0.1323,0.0627,0.0016
46,361259,32,1,0.1241,0.1733,0.0851,0.0016
82,361253,48,1,0.1036,0.2049,0.0947,0.0015
118,361242,81,1,0.1501,0.2569,0.0907,0.0014
13,361254,21,5,0.0113,0.1050,0.0068,0.0011
49,361259,32,5,0.0175,0.1423,0.0089,0.0013
85,361253,48,5,0.0227,0.1789,0.0095,0.0011
121,361242,81,5,0.0448,0.2351,0.0093,0.0011
16,361254,21,10,0.0081,0.0993,0.0028,0.0010
52,361259,32,10,0.0128,0.1332,0.0031,0.0010


In [10]:
display_df = df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5) & (df['n_estimators'] != 50)].sort_values(by=['n_estimators', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, rf_plus_fitting_time + lmdi_plus_values_time, lime_time, shap_values_time, local_mdi_time
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'rf_plus_fitting_time', 'lmdi_plus_rf_values_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_rf_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_rf_values_time'], inplace=True)
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'lmdi_plus_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lmdi_plus_time': 'LMDI+',
    'lime_rf_time': 'LIME',
    'shap_rf_values_time': 'TreeSHAP',
    'local_mdi_time': 'Local MDI'
})

# sort display_df by min samples leaf increasing and then number of features increasing
display_df = display_df.sort_values(by=['# of Estimators', '# of Features'])

# round to fourth decimal place
display_df = display_df.round(4)

In [11]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   # of Estimators |   LMDI+ |   LIME |   TreeSHAP |   Local MDI |
|-----------------:|----------------:|------------------:|--------:|-------:|-----------:|------------:|
|           361254 |              21 |               100 |  0.0113 | 0.105  |     0.0068 |      0.0011 |
|           361259 |              32 |               100 |  0.0175 | 0.1423 |     0.0089 |      0.0013 |
|           361253 |              48 |               100 |  0.0227 | 0.1789 |     0.0095 |      0.0011 |
|           361242 |              81 |               100 |  0.0448 | 0.2351 |     0.0093 |      0.0011 |
|           361254 |              21 |               500 |  0.06   | 0.2585 |     0.0345 |      0.0054 |
|           361259 |              32 |               500 |  0.0859 | 0.3133 |     0.04   |      0.0058 |
|           361253 |              48 |               500 |  0.1171 | 0.3416 |     0.0486 |      0.0054 |
|           361242 |              81 |               50

In [12]:
# get display_df in latex format
latex_df = display_df.to_latex(index=False, float_format="%.4f")
print(latex_df)

\begin{tabular}{rrrrrrr}
\toprule
OpenML Data ID & # of Features & # of Estimators & LMDI+ & LIME & TreeSHAP & Local MDI \\
\midrule
361254 & 21 & 100 & 0.0113 & 0.1050 & 0.0068 & 0.0011 \\
361259 & 32 & 100 & 0.0175 & 0.1423 & 0.0089 & 0.0013 \\
361253 & 48 & 100 & 0.0227 & 0.1789 & 0.0095 & 0.0011 \\
361242 & 81 & 100 & 0.0448 & 0.2351 & 0.0093 & 0.0011 \\
361254 & 21 & 500 & 0.0600 & 0.2585 & 0.0345 & 0.0054 \\
361259 & 32 & 500 & 0.0859 & 0.3133 & 0.0400 & 0.0058 \\
361253 & 48 & 500 & 0.1171 & 0.3416 & 0.0486 & 0.0054 \\
361242 & 81 & 500 & 0.2256 & 0.3800 & 0.0447 & 0.0051 \\
361254 & 21 & 1000 & 0.1721 & 0.4315 & 0.0677 & 0.0107 \\
361259 & 32 & 1000 & 0.2230 & 0.5056 & 0.0800 & 0.0114 \\
361253 & 48 & 1000 & 0.3602 & 0.5364 & 0.1018 & 0.0112 \\
361242 & 81 & 1000 & 0.4961 & 0.5497 & 0.0926 & 0.0106 \\
\bottomrule
\end{tabular}

